<a href="https://colab.research.google.com/github/velchan15/MachineLearning-InternshipStarter-FlyRank/blob/main/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

Turning the validated Week-5/6 model into a human-reviewed action playbook: ranked queue, reason codes, archetypes, limits, and monitoring triggers.


In [1]:
import pandas as pd
import numpy as np
import json, os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

np.random.seed(42)

csv_path = 'data/raw/content_refresh_anonymized.csv'
if os.path.exists(csv_path):
    os.remove(csv_path)
os.makedirs(os.path.dirname(csv_path), exist_ok=True)
import urllib.request
url = "https://raw.githubusercontent.com/velchan15/MachineLearning-InternshipStarter-FlyRank/main/data/raw/content_refresh_anonymized.csv"
urllib.request.urlretrieve(url, csv_path)

df = pd.read_csv(csv_path)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df['avg_position'] = df['avg_position'].replace(0, np.nan)

flagged_cols = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'avg_position']
for col in flagged_cols:
    df[f'has_{col}'] = df[col].notna().astype(int)
has_flag_cols = [f'has_{c}' for c in flagged_cols]

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d',
    'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct',
]
categorical_features = [
    'content_type', 'main_intent', 'provider_used', 'model_used',
    'competition_level', 'age_tier', 'freshness_tier',
    'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier',
]
X_cols = numeric_features + categorical_features + has_flag_cols
X = df[X_cols].copy()
y = df['is_declining_label'].copy()
groups = df['client_id'].copy()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('flag', 'passthrough', has_flag_cols),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order][:k].mean()

# Same validated split and model as Week 5/6 -- the queue below is built on the
# HELD-OUT test slice only, because that's the only slice where the precision
# numbers are honestly known.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
y_train, y_test = y.iloc[train_idx].reset_index(drop=True), y.iloc[test_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

lr = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))])
lr.fit(X_train, y_train)
proba_test = lr.predict_proba(X_test)[:, 1]
p10 = precision_at_k(proba_test, y_test, 10)
p50 = precision_at_k(proba_test, y_test, 50)
base_rate = y_test.mean()
print(f"Validated model (Week 5/6): P@10={p10:.3f}  P@50={p50:.3f}  base rate={base_rate:.3f}  (n={len(y_test)})")


Validated model (Week 5/6): P@10=0.800  P@50=0.840  base rate=0.511  (n=6163)


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
queue = df_test.copy()
queue['risk_score'] = proba_test
queue = queue.sort_values('risk_score', ascending=False).reset_index(drop=True)
queue['rank'] = queue.index + 1

median_days_visible = df_test['days_with_impressions'].median()

def build_reason(row):
    tags = []
    if row['freshness_tier'] in ['91-180', '181+']:
        tags.append('stale-content')
    if row['impression_tier'] in ['good', 'excellent']:
        tags.append('high-visibility')
    if row['days_with_impressions'] < median_days_visible:
        tags.append('inconsistent-visibility')
    if pd.notna(row['avg_position']) and row['avg_position'] > 20:
        tags.append('weak-ranking-position')
    if not tags:
        tags.append('general-decline-signal')
    return ' + '.join(tags)

queue['reason_code'] = queue.apply(build_reason, axis=1)

def archetype(row):
    high_traffic = row['impression_tier'] in ['good', 'excellent']
    stale = row['freshness_tier'] in ['91-180', '181+']
    high_risk = row['risk_score'] >= 0.5
    if high_risk and high_traffic:
        return 'High-Traffic Decliner'
    elif high_risk and stale and not high_traffic:
        return 'Stale & Fading'
    elif high_risk:
        return 'Early Warning'
    elif high_traffic:
        return 'Stable Performer'
    else:
        return 'Low-Priority / New'

ARCHETYPE_ACTION = {
    'High-Traffic Decliner': 'Priority refresh this quarter -- update content, re-check search intent match, verify technical SEO basics',
    'Stale & Fading': 'Refresh or consolidate -- if traffic is already low, consider merging into a stronger page instead of a full rewrite',
    'Early Warning': 'Add to next-cycle review queue -- not urgent, but track for two more reporting periods',
    'Stable Performer': 'No action -- maintain cadence, light monitoring only',
    'Low-Priority / New': 'No action -- too early or too low-traffic to prioritize; revisit next quarter',
}

queue['archetype'] = queue.apply(archetype, axis=1)
queue['recommended_action'] = queue['archetype'].map(ARCHETYPE_ACTION)

print("Archetype distribution (test portfolio, n=%d):" % len(queue))
print(queue['archetype'].value_counts())
print()
queue[['rank','content_id','risk_score','reason_code','archetype']].head(10)


Archetype distribution (test portfolio, n=6163):
archetype
Early Warning            2065
Low-Priority / New       1952
Stale & Fading            800
High-Traffic Decliner     786
Stable Performer          560
Name: count, dtype: int64



,rank,content_id,risk_score,reason_code,archetype
0,1,content_9532f197bbc8,1.000000,stale-content + high-visibility,High-Traffic Decliner
1,2,content_5fe46e04994d,1.000000,stale-content + high-visibility,High-Traffic Decliner
2,3,content_cea79ef51519,0.999999,high-visibility,High-Traffic Decliner
3,4,content_39881853ef0c,0.999999,high-visibility,High-Traffic Decliner
4,5,content_73c54f78c06a,0.999988,high-visibility,High-Traffic Decliner
5,6,content_8d3971bfd976,0.999986,high-visibility + weak-ranking-position,High-Traffic Decliner
6,7,content_3437133c7ccf,0.999951,high-visibility,High-Traffic Decliner
7,8,content_2dc625dcdf98,0.999448,high-visibility,High-Traffic Decliner
8,9,content_2db251d1a841,0.999382,high-visibility,High-Traffic Decliner
9,10,content_ca17a024f90c,0.998561,high-visibility,High-Traffic Decliner


**Archetype → action mapping:**

| Archetype | Meaning | Recommended action |
|---|---|---|
| High-Traffic Decliner | High visibility + high model risk score | Priority refresh this quarter |
| Stale & Fading | Old content, high risk, but not high-traffic | Refresh or consolidate into a stronger page |
| Early Warning | High risk, but not yet high- or stale-traffic | Add to next review cycle, not urgent |
| Stable Performer | High traffic, low risk | No action, light monitoring |
| Low-Priority / New | Low traffic, low risk | No action, revisit later |

**Decay/refresh insight:**

In [3]:
decay_by_freshness = df.groupby('freshness_tier')['is_declining_label'].agg(['mean','count'])
decay_by_freshness.columns = ['decline_rate', 'n_pages']
decay_by_freshness


,decline_rate,n_pages
freshness_tier,,
0-30,0.511377,20480
181+,0.471264,174
31-90,0.588571,175
91-180,0.611057,9171


Pages in the 91-180 day freshness tier show the highest observed decline rate (61.1%), noticeably above fresh content (0-30 days, 51.1%). This is the core "decay" pattern the playbook acts on: content tends to start showing decline signals a few months after its last update, well before the 6-month mark.

One honest caveat: the 181+ day tier shows a *lower* decline rate (47.1%) than the 91-180 day tier, which on the surface looks backwards. This is very likely **survivorship bias** — pages still being tracked at 181+ days without having been retired or de-indexed are probably the ones that already stabilized; pages that kept declining past 180 days may have already been pulled or deprioritized out of this portfolio snapshot. This nuance is noted here rather than smoothed over.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** This playbook is a **prioritization aid** for a content strategist or SEO lead deciding which pages in a portfolio to review first in a given refresh cycle. It answers "what should a human look at first," not "what should change automatically."

**Who should use it:** Content/SEO team members doing quarterly or monthly refresh planning, with enough context on the brand and content to judge whether a flagged page is actually worth acting on.

**Where it stops being valid:**
- **Single snapshot, no causal claims.** The model is trained on one trailing-90-day cross-section. It **observes** patterns associated with decline; it does not know that refreshing a page will fix anything — that would require a controlled before/after test this data can't provide.
- **Precision is a portfolio-level number, not a per-page guarantee.** Precision@50 = 0.84 means: of the top 50 pages by risk score, about 84% were actually declining *in this test sample*. It does not mean any individual flagged page has an 84% chance of being a true decliner — some of the 50 are wrong, and the model can't say in advance which ones.
- **Doesn't generalize to a brand-type or content-type the model hasn't seen much of.** With only 32 clients in the training data, a genuinely new kind of brand or content mix is outside what this model has learned from.
- **Not validated against actual refresh outcomes.** No one has yet refreshed the flagged pages and measured whether it helped — this playbook is decision-support for *what to review first*, not proof that reviewing it will improve anything.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a human must check:**
1. **Is the page still strategically relevant?** (product discontinued, campaign ended, etc. — the model has no idea)
2. **Is the decline explainable by something outside content quality** (seasonality, a known algorithm update, a broken tracking tag)?
3. **Does the reason code make sense for this specific page**, or does it look like a data quirk (e.g. a tracking gap misread as "inconsistent visibility")?
4. **Is there a legal, compliance, or brand-safety reason this page can't simply be rewritten** (regulated claims, legal disclaimers, translated content)?

**What should NEVER be automated based on this playbook alone:**
- **Auto-publishing rewritten content** without a human reading it first — the model has no sense of brand voice, accuracy, or claim safety.
- **Auto-deleting or de-indexing pages** flagged as low-priority — "low priority for refresh" is not the same as "safe to remove."
- **Using the risk score as a performance metric for content creators** — it's a decline signal, not a quality judgment on who wrote the page or when.
- **Triggering client-facing communication** ("your content is declining") directly from the score — that needs human framing and context every time.

The playbook's job stops at *surfacing and ranking*. Every action from there is a human decision.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Monitoring, on a light cadence (recommend monthly):**
- **Re-check precision@50 on a fresh held-out slice** of newly-arrived data. If it drops meaningfully below the current 0.84 (e.g. below ~0.65), the model's signal has weakened and should not be trusted for prioritization until investigated.
- **Watch the base rate.** The current test base rate is 0.511 (about half the portfolio is "declining" at any time). A large shift in base rate (e.g. a client-wide traffic collapse from an algorithm update) means the whole population changed, and old precision numbers no longer describe the current portfolio.
- **Watch archetype distribution drift.** If "High-Traffic Decliner" suddenly balloons or nearly disappears from one cycle to the next, that's worth a manual look before trusting the new queue outright.

**Retrain triggers:**
- A new client/brand type is onboarded that looks meaningfully different from the 32 in the training data (e.g. a different content_type mix).
- Three consecutive monitoring cycles show precision@50 below the ~0.65 floor.
- FlyRank's own trend/label definition changes (e.g. the window used for "declining" is redefined) — the label itself would no longer match what this model was trained on.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [4]:
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

export_cols = ['rank','content_id','client_id','risk_score','reason_code','archetype','recommended_action',
               'freshness_tier','impression_tier','avg_position','is_declining_label']
queue[export_cols].to_csv('work/outputs/w07_action_queue.csv', index=False)
print(f"Exported queue: work/outputs/w07_action_queue.csv ({len(queue)} rows)")

metrics = {
    'model': 'Logistic Regression (Week 5/6 validated model)',
    'split_design': 'grouped by client_id, 80/20',
    'test_set_size': int(len(y_test)),
    'test_base_rate': round(float(base_rate), 3),
    'precision_at_10': round(float(p10), 3),
    'precision_at_50': round(float(p50), 3),
    'archetype_counts': queue['archetype'].value_counts().to_dict(),
    'decay_by_freshness_tier': {k: round(v, 3) for k, v in decay_by_freshness['decline_rate'].to_dict().items()},
    'notes': 'Metrics computed on the held-out client-grouped test split (n=6163), not on training data. Queue scored on the same held-out slice to keep the playbook honest about where the precision numbers actually apply.'
}
with open('work/outputs/w07_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Exported metrics: work/outputs/w07_metrics.json")


Exported queue: work/outputs/w07_action_queue.csv (6163 rows)
Exported metrics: work/outputs/w07_metrics.json


In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8,5))
counts = queue['archetype'].value_counts().sort_values()
ax.barh(counts.index, counts.values, color='#3E7C74')
ax.set_xlabel('Number of pages (test portfolio, n=6,163)')
ax.set_title('Content Action Playbook — Archetype Distribution')
plt.tight_layout()
plt.savefig('work/figures/w07_archetype_distribution.png', dpi=150)
plt.show()
print("Exported figure: work/figures/w07_archetype_distribution.png")


Exported figure: work/figures/w07_archetype_distribution.png


**Note on the CSV:** `work/outputs/w07_action_queue.csv` is intentionally left out of git by the repo's CI leak-guard, which blocks data files from being committed. This notebook regenerates it on every run, so nothing is lost — only the figure (`work/figures/`) and the metrics JSON (`work/outputs/w07_metrics.json`) get committed, since those are the receipts the paper's numbers trace back to, and they contain no row-level data.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (content_id/client_id are pseudonyms only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
